# Simple RAG LLM Usage for Document Reading and Information Retrieval

This notebook implements a small, fully local Retrieval-Augmented Generation (RAG) pipeline for PDF question answering.

Pipeline:

**PDF → text extraction → chunking → embeddings → FAISS retrieval → local LLM generation**

The embedding model is `BAAI/bge-small-en-v1.5` and the local LLM is `qwen3:4b` served through Ollama.

## 1. Load a PDF

Place a PDF at `../data/documents/document.pdf`.

PyMuPDF extracts text page by page while preserving page metadata for later source attribution.

In [ ]:
from pathlib import Path
import pymupdf

pdf_path = Path("../data/documents/document.pdf")
source_name = pdf_path.name

document = pymupdf.open(pdf_path)
print("Number of pages:", len(document))

In [ ]:
all_pages = []

for page_number, page in enumerate(document):
    all_pages.append({
        "page": page_number + 1,
        "text": page.get_text()
    })

print("Pages extracted:", len(all_pages))

## 2. Split the document into chunks

RAG does not send the whole PDF to the model. It splits the text into smaller overlapping chunks so retrieval can return only the most relevant context.

Here we use LangChain's `RecursiveCharacterTextSplitter` with:

- chunk size: 1000 characters
- overlap: 200 characters

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

page_documents = [
    Document(
        page_content=page["text"],
        metadata={"page": page["page"], "source": source_name}
    )
    for page in all_pages
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(page_documents)
print("Total chunks:", len(chunks))

## 3. Create embeddings

An embedding converts text into a numerical vector that represents semantic meaning.

We use `BAAI/bge-small-en-v1.5`. Normalized embeddings allow inner-product search to behave like cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

chunk_texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
).astype("float32")

print("Embedding matrix shape:", embeddings.shape)

## 4. Build a FAISS vector index

FAISS stores the chunk vectors and retrieves the nearest vectors to a query embedding.

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vectors stored:", index.ntotal)

## 5. Create the retriever

The retriever embeds the user query, searches FAISS, and maps the returned vector IDs back to document chunks and metadata.

In [ ]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, chunk_index in zip(scores[0], indices[0]):
        chunk = chunks[chunk_index]

        results.append({
            "text": chunk.page_content,
            "page": chunk.metadata["page"],
            "source": chunk.metadata["source"],
            "score": float(score)
        })

    return results

In [ ]:
results = retrieve("What dataset is used?", k=3)

for rank, result in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Page:", result["page"])
    print(result["text"][:500])

## 6. Add the local LLM: Generation

Ollama runs `qwen3:4b` locally.

The retrieved chunks are inserted into the prompt as context. The LLM is instructed to answer only from that context.

In [ ]:
from ollama import chat

def rag_answer(question, k=3):
    results = retrieve(question, k=k)

    context_parts = []
    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"[Source {i}: {result['source']}, page {result['page']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a question-answering assistant.

Use only the provided context to answer the question.
If the answer is not contained in the context, say:
"I don't know based on the provided documents."

When useful, mention the supporting page number.

Context:
{context}

Question:
{question}

Answer:
"""

    response = chat(
        model="qwen3:4b",
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "answer": response.message.content,
        "sources": [
            {
                "source": r["source"],
                "page": r["page"],
                "score": r["score"]
            }
            for r in results
        ]
    }

In [ ]:
result = rag_answer("What dataset is used in the paper?")

print(result["answer"])
print("\nRetrieved sources:")
for source in result["sources"]:
    print(
        f"- {source['source']}, page {source['page']} "
        f"(similarity={source['score']:.3f})"
    )

## What this project demonstrates

- PDF document ingestion
- text chunking with overlap
- semantic embeddings
- cosine-style vector similarity
- FAISS nearest-neighbor retrieval
- Retrieval-Augmented Generation
- local LLM inference with Ollama
- source/page metadata preservation

This is intentionally a simple educational RAG implementation. The next improvements would be multi-document ingestion, reranking, evaluation, persistent vector storage, and an API/UI.